# 10 . Scenario Lab — versioned, composable setups

Back-test a **hand-described setup** and store it by its settings, so it re-runs on any symbol/TF and
tweaks into variants. A run is four named, **versioned** pieces:

> **Scenario** (entry logic) × **Stop** × **TP** × **Secondary** (post-entry rule) × **params**

bundled into a **Preset** (JSON) that reproduces the exact backtest.

The worked setup is a **support-break short**: price was above a smoothed 200-EMA (positive gradient),
made HH/HL into resistance with lows trapped on a support, rolled over below the EMA (even full candles
below — *still no sell*), then **broke through the support** (**level B**) → short.

### Where do I set … ?

| I want to change… | Param | Section |
|---|---|---|
| Which support level counts, and how it must break | `break_mode`, `sr_min_touches`, `prev_low_n` | 2 |
| **N** candles for entry confirmation | `confirm_red`, `n_red`, `wait_n`, `pullback_lookback` | 2 |
| **Normal vs Heikin-Ashi** candles (everywhere) | `candle_src='real'\|'ha'` | 2 |
| The **gradient threshold** on the 200-EMA | `grad_max` (`None` = off) | 2 |
| Which EMA the setup uses | `grad_period`, `grad_smooth`, `grad_lookback` | 2 |
| Where the **stop** goes | `STOP` + `stop_*` params | 2, 8 |
| How profit is taken | `TP` + `tp_*` params | 2, 10 |
| Bail-out after a shaky move | `SECONDARY`, `sec_*` | 2, 9 |
| **Block certain hours / days / times** | `block_hours`, `block_weekdays`, `block_times` | 2 |


> Logic lives in `quant/scenarios/`, charting in `quant/viz/studies.py`; this notebook is
> declaration-only. Tuned configs save to `scenarios/presets/`.

## 0 . Setup

In [1]:
import sys, os, warnings; warnings.filterwarnings('ignore')
try:
    import quant
except ModuleNotFoundError:
    sys.path.insert(0, os.path.abspath('..')); import quant
import numpy as np, pandas as pd
from quant.data import get_ohlcv
from quant import scenarios as sc
from quant.scenarios import Preset
from quant.viz import scenario_figure, zoom_to_trade, open_in_browser, CROSSHAIR_JS
pd.set_option('display.max_rows', 120); pd.set_option('display.width', 200)
print('scenarios  :', sorted(sc.SCENARIOS))
print('stops      :', sorted(sc.STOP_METHODS))
print('tps        :', sorted(sc.TP_METHODS))
print('secondaries:', sorted(sc.SECONDARIES))

scenarios  : ['support_break_short@v1.0', 'support_break_short@v1.1', 'support_break_short@v1.2', 'support_break_short@v1.3']
stops      : ['atr@v1', 'high_of_n@v1', 'leg_high@v1', 'pct@v1', 'prev_high@v1', 'swing_high@v1', 'zone_top@v1']
tps        : ['back_above_ema@v1', 'fixed_pct@v1', 'n_green_exit@v1', 'next_zone@v1', 'none@v1', 'opposite_candle@v1', 'rr@v1', 'rr_partials@v1', 'trailing@v1']
secondaries: ['profit_candle_bail@v1', 'run_past_tp@v1', 'shaky_trend_bail@v1']


## 1 . Data — recent BTC 1m
Defaults to the window from the reference screenshot. Switch `SYMBOL/SOURCE` for gold
(`'XAUUSD','dukascopy','cfd'`) or a PAXG proxy.

In [2]:
SYMBOL, SOURCE, MARKET, TF = 'BTCUSDT', 'binance', 'spot', '1m'
START, END = '2026-07-28', '2026-07-30'#'2026-07-28 07:00'
df = get_ohlcv(SYMBOL, TF, start=START, end=END, source=SOURCE, market=MARKET, tz='UTC')
print(f'{len(df):,} {TF} bars   {df.t.min()}  ->  {df.t.max()}')

[17:51:26] INFO | quant.data | cache HIT  binance BTCUSDT 1m -> pushdown load


2,881 1m bars   2026-07-28 00:00:00+00:00  ->  2026-07-30 00:00:00+00:00


## 2 . The config — every knob, in one place
Change anything here and re-run the notebook; everything downstream follows.

In [12]:
# ---- the four pieces -------------------------------------------------------------------------
SCENARIO  = 'support_break_short@v1.2'  # v1.0 core | v1.1 +structure | v1.2 no-structure | v1.3 nearest-trough
STOP      = 'leg_high@v1'               # pct | swing_high | prev_high | leg_high | high_of_n | zone_top | atr
TP        = 'rr@v1'                     # rr | fixed_pct | trailing | opposite_candle | n_green_exit |
                                        #   back_above_ema | next_zone | rr_partials
# SECONDARY = post-entry rules. A LIST -> all active at once; they do NOT conflict (each just adds
# another way for an open trade to become ARMED, and they share the in-profit-gated exit).
# None = off.  Toggle each independently by commenting it out.
SECONDARY = [
    'shaky_trend_bail@v1',   # price gets back to the EMA -> bail on the next opposite candle, if green
    'run_past_tp@v1',        # reaching the TP ARMS instead of closing -> ride on, exit on N opposite candles
]

PARAMS = dict(
    # -- ENTRY: when the break counts as a trigger -------------------------------------------
    entry_mode='break_close',   # break_close | wait_n | n_red | pullback
    break_mode='close_below',   # 'close_below' = candle CLOSES below level B (default)
                                # 'full_below'  = whole candle forms below level B
    prev_low_n=20,              # v1.2: level B = lowest low of the prior N bars
    sr_min_touches=1,           # v1.0/v1.1: min touches for a support zone to count
    break_min_ema_dist_pct=None,  # optional: reject level B if it sits TOO CLOSE to the EMA line (%)
    break_max_ema_dist_pct=None,  # optional: reject level B if it sits TOO FAR from the EMA line (%)

    # -- CANDLES: type + the N counts ---------------------------------------------------------
    candle_src='real',          # 'real' or 'ha'  -> ALL candle colour/shape tests use this source
    confirm_red=2,              # v1.2 context: N red candles below the EMA
    n_red=1,                    # entry_mode='n_red'   : N red candles after the break
    wait_n=3,                   # entry_mode='wait_n'  : N bars to wait after the break
    pullback_lookback=5,        # entry_mode='pullback': N bar window for the retest

    # -- EMA + GRADIENT THRESHOLD --------------------------------------------------------------
    grad_period=200, grad_smooth=9,   # the "200 EMA smoothed by 9" line
    grad_lookback=5,                  # bars used to measure its slope
    grad_max=0.0,               # THRESHOLD: entry needs grad < this. 0.0 = any down-slope,
                                # -0.05 = must be clearly falling, None = no gradient gate.

    # -- STOP ------------------------------------------------------------------------------------
    stop_buffer_pct=0.05,       # buffer beyond the structural level (%)
    stop_max_risk_pct=0.0,      # cap: if the structural stop risks more than this %, use stop_pct
    stop_pct=0.6,               # 'pct' stop, and the fallback for structural stops
    stop_above_ema_only=False,  # leg_high: count only highs that poked ABOVE the EMA line
    stop_lr=3,                  # prev_high: pivot look-around (NOTE: costs `lr` bars of lag)
    stop_swing_lr=12,           # swing_high: pivot look-around
    stop_n=20,                  # high_of_n: N bars
    stop_atr_period=14, stop_atr_mult=1.5,   # atr stop

    # -- TAKE PROFIT -------------------------------------------------------------------------
    tp_rr=2.0,                  # rr / rr_partials: R multiple
    tp_pct=0.6,                 # fixed_pct
    trail_pct=1.0,              # trailing
    tp_opp_n=1,                 # opposite_candle ("TP L"): close after N opposite-colour candles
    tp_zone_pct=0.1,            # next_zone

    # -- WHEN TO TRADE: block-lists (exclude) and allow-lists (keep). All optional, all None=off.
    #    All times are the tz of the `t` column (UTC here); set filter_tz='Asia/Karachi' to use local.
    block_hours=None,        # e.g. (12, 16, 18)  -> NO entries in those clock hours (any set, non-contiguous)
    block_weekdays=None,     # e.g. (5, 6)        -> NO entries Sat/Sun     (0=Mon .. 6=Sun)
    block_times=('08:00','13:00'),        # e.g. (('13:30','14:15'), ('20:00','20:30')) -> minute-level blackouts
    hours=None,              # ALLOW-list: (7, 16) keeps ONLY 07:00-16:00 (one contiguous window)
    weekdays=None,           # ALLOW-list: (0,1,2,3,4) keeps ONLY Mon-Fri
    session=None,            # ALLOW-list: 'london' | 'newyork' | 'tokyo' | 'sydney'
    filter_tz='UTC',          # evaluate all of the above in this tz instead of the data's

    # -- SECONDARY (only used when SECONDARY is set) -------------------------------------------
    sec_touch=True,             # arm on a TOUCH of the EMA line as well as a close above it
    sec_opp_n=1,                # bail after N opposite-colour candles
    sec_replace_tp=False,
    sec_arm_on_tp=True,         # run_past_tp: reaching the TP ARMS the trade instead of closing it       # False = keep the main TP too (first one wins); True = TP is dropped
    sec_arm_mode='near', sec_arm_tol_pct=0.05,   # high came within 0.05% BELOW the line
    # sec_arm_mode='touch',                        # high >= line              (default, = old behaviour)
    # sec_arm_mode='close_above',                  # candle CLOSED above
    # sec_arm_mode='full_above',                   # WHOLE candle above (low > line)
)

# ---- CHART layers (used by the chart in section 5) --------------------------------------------
SHOW_ATR_BANDS = None #(1, 2, 3, 4)   # EMA +/- k*ATR envelopes -> SEE how extended an entry was. None=off
ATR_PERIOD     = 14
SHOW_RSI       = False           # adds an RSI pane
RSI_PERIOD     = 14
SHOW_RSI_DIV   = False           # draw RSI divergences on BOTH the price and RSI panes
SHOW_STRUCTURE = True           # HH/HL/LH/LL pivot labels + trend ribbon (up / down / sideways)
STRUCT_LR      = 10             # pivot look-around; smaller = faster labels but noisier

CFG = dict(initial_cash=10_000, spread=0.0, sizing_mode='risk_pct_equity', sizing_value=1.0)

run = sc.run_pieces(df, SCENARIO, STOP, TP, secondary=SECONDARY, params=PARAMS, cfg=CFG)
print(sc.resolve_scenario(SCENARIO).description)
print('stop :', sc.resolve_stop(STOP).description)
print('tp   :', sc.resolve_tp(TP).description)
s = run.stats
print(f"\ntrades {int(s['num_trades'])} | win {s['win_rate_pct']:.0f}% | ret {s['total_return_pct']:+.2f}% "
      f"| dd {s['max_drawdown_pct']:.2f}% | PF {s['profit_factor']:.2f} | Sharpe {s['sharpe']:.2f} "
      f"| kernel {run.result.elapsed_s*1e3:.0f} ms")

[17:56:08] INFO | quant.engine | backtest done | bars=2,881 trades=8 final_cash=10151.04 return=1.51% dd=0.54% | 0.3 ms


No-structure: no S/R zones — N consecutive red candles below the EMA and a break of the lowest low of the prior N bars.
stop : Stop above the high of the CURRENT down-leg — the running max since price last closed below the EMA line. No confirmation lag, so it tracks the high a human would draw; set stop_above_ema_only to use only highs that poked above the EMA.
tp   : Fixed reward:risk target (R multiple of the stop distance).

trades 8 | win 88% | ret +1.51% | dd 0.54% | PF 4.89 | Sharpe 19.47 | kernel 0 ms


## 3 . Trades

In [13]:
run.trades[['side','entry_time','exit_time','entry_price','stop_price','exit_price',
            'pnl','return_pct','bars_held','close_reason']]

,side,entry_time,exit_time,entry_price,stop_price,exit_price,pnl,return_pct,bars_held,close_reason
0,short,2026-07-28 03:57:00+00:00,2026-07-28 04:31:00+00:00,63284.60,63373.050690,63269.300000,2.417650,0.024176,34,alt_exit
1,short,2026-07-28 13:23:00+00:00,2026-07-28 13:40:00+00:00,63393.07,63509.728995,63063.430000,52.011946,0.519994,17,alt_exit
2,short,2026-07-28 17:37:00+00:00,2026-07-28 18:00:00+00:00,63522.73,63767.878005,63767.878005,-38.802226,-0.385922,23,stop_loss
3,short,2026-07-28 22:45:00+00:00,2026-07-28 23:10:00+00:00,63860.00,63949.959000,63658.000000,31.681126,0.316317,25,alt_exit
4,short,2026-07-29 01:43:00+00:00,2026-07-29 02:18:00+00:00,63796.80,63971.339685,63750.000000,7.370496,0.073358,35,alt_exit
5,short,2026-07-29 03:27:00+00:00,2026-07-29 03:43:00+00:00,63834.00,63971.970000,63764.210000,10.992826,0.109330,16,alt_exit
6,short,2026-07-29 14:25:00+00:00,2026-07-29 16:24:00+00:00,64212.94,64456.212000,64089.990000,19.272974,0.191472,119,alt_exit
7,short,2026-07-29 19:28:00+00:00,2026-07-29 20:03:00+00:00,63980.01,64222.105005,63560.690000,66.095942,0.655392,35,alt_exit


## 4 . Level B audit — what exactly did each trade break?
`break_level` is latched at the **first break of the episode** and held, so it always names the level
that actually broke (an S/R support steps down to the next zone the moment price closes through it).
`ema_dist_%` is what the `break_min/max_ema_dist_pct` filters act on — use this table to pick them.

In [14]:
d = run.df
rows = []
for i, t in run.trades.iterrows():
    ei = int(t.entry_i); B = d.break_level.iloc[ei]; line = d.grad_line.iloc[ei]
    rows.append(dict(trade=i, entry_time=t.entry_time, level_B=round(B, 2),
                     entry=round(t.entry_price, 2), stop=round(t.stop_price, 2),
                     ema_line=round(line, 2),
                     ema_dist_pct=round(abs(B - line) / line * 100, 4),
                     risk=round(t.stop_price - t.entry_price, 2), reason=t.close_reason))
pd.DataFrame(rows)

,trade,entry_time,level_B,entry,stop,ema_line,ema_dist_pct,risk,reason
0,0,2026-07-28 03:57:00+00:00,63303.35,63284.60,63373.05,63339.40,0.0569,88.45,alt_exit
1,1,2026-07-28 13:23:00+00:00,63459.86,63393.07,63509.73,63476.47,0.0262,116.66,alt_exit
2,2,2026-07-28 17:37:00+00:00,63694.70,63522.73,63767.88,63717.67,0.0361,245.15,stop_loss
3,3,2026-07-28 22:45:00+00:00,63918.00,63860.00,63949.96,63910.19,0.0122,89.96,alt_exit
4,4,2026-07-29 01:43:00+00:00,63830.35,63796.80,63971.34,63910.64,0.1256,174.54,alt_exit
5,5,2026-07-29 03:27:00+00:00,63862.73,63834.00,63971.97,63906.32,0.0682,137.97,alt_exit
6,6,2026-07-29 14:25:00+00:00,64316.11,64212.94,64456.21,64399.85,0.1300,243.27,alt_exit
7,7,2026-07-29 19:28:00+00:00,64180.38,63980.01,64222.11,64174.77,0.0087,242.10,alt_exit


## 5 . The annotated chart (opens in a browser tab)
Candles + the red-dotted **EMA line**, **S/R zones**, and per trade the **entry** (red ▼), its **stop**
(red dotted), **level B** (amber dashed — the level it broke) and the **exit** coloured by reason, plus
the **gradient pane**. Every element is its own legend entry — click to toggle.

In [15]:
# no explicit title -> scenario_figure names every active piece, incl. the secondaries
fig = scenario_figure(run, atr_bands=SHOW_ATR_BANDS, atr_period=ATR_PERIOD,
                      rsi=SHOW_RSI, rsi_period=RSI_PERIOD, rsi_divergence=SHOW_RSI_DIV,
                      structure=SHOW_STRUCTURE, struct_lr=STRUCT_LR,
                      height=1150 if SHOW_RSI else 900)
print('chart ->', open_in_browser(fig, 'scenario_chart.html', extra_js=CROSSHAIR_JS))

chart -> C:\Users\Talal Zahid\Desktop\Talal\Binance\notebooks\scenario_chart.html


## 6 . Zoom to one trade

In [7]:
if len(run.trades):
    TRADE_I = run.trades.index[0]
    fig1 = scenario_figure(run)
    zoom_to_trade(fig1, run, TRADE_I, pad_bars=60)
    print('zoom ->', open_in_browser(fig1, 'scenario_trade.html', extra_js=CROSSHAIR_JS))

zoom -> C:\Users\Talal Zahid\Desktop\Talal\Binance\notebooks\scenario_trade.html


## 7 . Q1 — where should the stop go?
Same entries, different stop methods. **Watch the lag:** `swing_high`/`prev_high` are *pivot* levels —
a pivot needs `lr` bars of confirmation to its right, so the level is always ≥`lr` bars old and can sit
far above the entry. `leg_high` (running max since price went below the EMA) and `high_of_n` have **no
confirmation lag**.

In [8]:
rows = []
for stop, extra in [('pct@v1', {}), ('swing_high@v1', {}), ('prev_high@v1', {}),
                    ('leg_high@v1', {}), ('leg_high@v1', dict(stop_above_ema_only=True)),
                    ('high_of_n@v1', dict(stop_n=10)), ('zone_top@v1', {}), ('atr@v1', {})]:
    r = sc.run_pieces(df, SCENARIO, stop, TP, params={**PARAMS, **extra}, cfg=CFG)
    st = r.stats
    for i, t in r.trades.iterrows():
        rows.append(dict(stop=stop.split('@')[0] + (' aboveEMA' if extra.get('stop_above_ema_only') else ''),
                         entry=round(t.entry_price, 2), stop_px=round(t.stop_price, 2),
                         risk=round(t.stop_price - t.entry_price, 2),
                         reason=t.close_reason, pnl=round(t.pnl, 2)))
pd.DataFrame(rows)

[17:51:37] INFO | quant.engine | backtest done | bars=2,881 trades=6 final_cash=9696.00 return=-3.04% dd=3.50% | 0.6 ms
[17:51:37] INFO | quant.engine | backtest done | bars=2,881 trades=6 final_cash=9924.44 return=-0.76% dd=2.19% | 1.0 ms
[17:51:37] INFO | quant.engine | backtest done | bars=2,881 trades=8 final_cash=10015.66 return=0.16% dd=1.23% | 0.7 ms
[17:51:37] INFO | quant.engine | backtest done | bars=2,881 trades=8 final_cash=10114.72 return=1.15% dd=0.85% | 0.5 ms
[17:51:37] INFO | quant.engine | backtest done | bars=2,881 trades=8 final_cash=10114.72 return=1.15% dd=0.85% | 0.7 ms
[17:51:37] INFO | quant.engine | backtest done | bars=2,881 trades=8 final_cash=10139.19 return=1.39% dd=0.98% | 0.5 ms
[17:51:37] INFO | quant.engine | backtest done | bars=2,881 trades=9 final_cash=10164.16 return=1.64% dd=0.38% | 0.5 ms
[17:51:37] INFO | quant.engine | backtest done | bars=2,881 trades=9 final_cash=10051.13 return=0.51% dd=0.27% | 0.4 ms


,stop,entry,stop_px,risk,reason,pnl
0,pct,63284.60,63664.31,379.71,stop_loss,-60.00
1,pct,63393.07,63773.43,380.36,stop_loss,-59.64
2,pct,63522.73,63903.87,381.14,stop_loss,-59.28
3,pct,63860.00,64243.16,383.16,stop_loss,-58.93
4,pct,64212.94,64598.22,385.28,stop_loss,-58.57
5,pct,63980.01,64363.89,383.88,forced_close_end,-7.58
6,swing_high,63284.60,63407.08,122.48,stop_loss,-19.35
7,swing_high,63393.07,63619.79,226.72,take_profit,71.39
8,swing_high,63522.73,64132.05,609.32,stop_loss,-96.42
9,swing_high,63796.80,63977.97,181.17,stop_loss,-28.27


## 8 . The secondary rule — bail out of a shaky trend
*"Once the trade is open, if price closes above / touches the EMA line the trend is shaky — so close on
the first opposite-colour candle, **provided we're in profit**. The stop is unchanged."*

This is **per-trade** state (armed only from *this* trade's own entry), so the engine tracks it; such
closes are tagged `alt_exit`. `sec_replace_tp` decides whether the original TP stays live alongside it.

In [9]:
rows = []
for tag, secondary, extra in [('off', None, {}),
                              ('on (keep TP)', 'shaky_trend_bail@v1', dict(sec_replace_tp=False)),
                              ('on (replace TP)', 'shaky_trend_bail@v1', dict(sec_replace_tp=True))]:
    r = sc.run_pieces(df, SCENARIO, STOP, TP, secondary=secondary,
                      params={**PARAMS, **extra}, cfg=CFG)
    st = r.stats
    rows.append(dict(secondary=tag, trades=int(st['num_trades']),
                     ret=round(st['total_return_pct'], 3), win=round(st['win_rate_pct']),
                     sharpe=round(st['sharpe'], 2),
                     reasons=dict(r.trades.close_reason.value_counts()) if len(r.trades) else {}))
pd.DataFrame(rows)

[17:51:37] INFO | quant.engine | backtest done | bars=2,881 trades=8 final_cash=10114.72 return=1.15% dd=0.85% | 0.5 ms
[17:51:37] INFO | quant.engine | backtest done | bars=2,881 trades=8 final_cash=10151.04 return=1.51% dd=0.54% | 1.2 ms
[17:51:37] INFO | quant.engine | backtest done | bars=2,881 trades=8 final_cash=10151.04 return=1.51% dd=0.54% | 0.4 ms


,secondary,trades,ret,win,sharpe,reasons
0,off,8,1.147,50,13.66,"{'stop_loss': 4, 'take_profit': 4}"
1,on (keep TP),8,1.510,88,19.47,"{'alt_exit': 7, 'stop_loss': 1}"
2,on (replace TP),8,1.510,88,19.47,"{'alt_exit': 7, 'stop_loss': 1}"


## 9 . Q2 / Q3 — entry timing and TP regime
Each row is a full backtest. Copy a winning combo up into section 2 to chart it.

In [10]:
def one(scen, stop, tp, secondary=None, **over):
    r = sc.run_pieces(df, scen, stop, tp, secondary=secondary, params={**PARAMS, **over}, cfg=CFG)
    s = r.stats
    return dict(scenario=scen.split('@')[1], stop=stop.split('@')[0], tp=tp.split('@')[0],
                entry=over.get('entry_mode', PARAMS['entry_mode']),
                brk=over.get('break_mode', PARAMS['break_mode']),
                trades=int(s['num_trades']), win=round(s['win_rate_pct']),
                ret=round(s['total_return_pct'], 2), dd=round(s['max_drawdown_pct'], 2),
                pf=round(s['profit_factor'], 2), sharpe=round(s['sharpe'], 2))

rows = []
for m in ['break_close', 'wait_n', 'n_red', 'pullback']:            # Q2 entry timing
    rows.append(one(SCENARIO, STOP, TP, entry_mode=m))
for bm in ['close_below', 'full_below']:                            # the level-B rule
    rows.append(one(SCENARIO, STOP, TP, break_mode=bm))
for tp in ['rr@v1','fixed_pct@v1','trailing@v1','opposite_candle@v1','next_zone@v1','rr_partials@v1']:
    rows.append(one(SCENARIO, STOP, tp))                            # Q3 TP regime
for v in ['v1.0','v1.1','v1.2','v1.3']:                             # scenario versions
    rows.append(one(f'support_break_short@{v}', STOP, TP))
pd.DataFrame(rows)

[17:51:37] INFO | quant.engine | backtest done | bars=2,881 trades=8 final_cash=10114.72 return=1.15% dd=0.85% | 0.7 ms
[17:51:37] INFO | quant.engine | backtest done | bars=2,881 trades=14 final_cash=9948.41 return=-0.52% dd=1.05% | 0.4 ms
[17:51:37] INFO | quant.engine | backtest done | bars=2,881 trades=16 final_cash=9929.65 return=-0.70% dd=1.10% | 0.5 ms
[17:51:37] INFO | quant.engine | backtest done | bars=2,881 trades=19 final_cash=9940.95 return=-0.59% dd=1.06% | 0.4 ms
[17:51:37] INFO | quant.engine | backtest done | bars=2,881 trades=8 final_cash=10114.72 return=1.15% dd=0.85% | 0.7 ms
[17:51:37] INFO | quant.engine | backtest done | bars=2,881 trades=9 final_cash=10046.02 return=0.46% dd=0.86% | 0.5 ms
[17:51:37] INFO | quant.engine | backtest done | bars=2,881 trades=8 final_cash=10114.72 return=1.15% dd=0.85% | 0.5 ms
[17:51:37] INFO | quant.engine | backtest done | bars=2,881 trades=8 final_cash=10063.88 return=0.64% dd=1.24% | 0.5 ms
[17:51:37] INFO | quant.engine | back

,scenario,stop,tp,entry,brk,trades,win,ret,dd,pf,sharpe
0,v1.2,leg_high,rr,break_close,close_below,8,50,1.15,0.85,2.13,13.66
1,v1.2,leg_high,rr,wait_n,close_below,14,29,-0.52,1.05,0.72,-6.87
2,v1.2,leg_high,rr,n_red,close_below,16,25,-0.70,1.10,0.64,-9.43
3,v1.2,leg_high,rr,pullback,close_below,19,32,-0.59,1.06,0.71,-8.49
4,v1.2,leg_high,rr,break_close,close_below,8,50,1.15,0.85,2.13,13.66
5,v1.2,leg_high,rr,break_close,full_below,9,44,0.46,0.86,1.29,5.46
6,v1.2,leg_high,rr,break_close,close_below,8,50,1.15,0.85,2.13,13.66
7,v1.2,leg_high,fixed_pct,break_close,close_below,8,38,0.64,1.24,1.55,7.48
8,v1.2,leg_high,trailing,break_close,close_below,8,25,-1.06,2.18,0.13,-9.00
9,v1.2,leg_high,opposite_candle,break_close,close_below,9,22,0.03,0.10,1.14,2.61


## 10 . Save the config as a versioned Preset
Reload later on any symbol/TF with `Preset.from_json(...)` + `sc.run(preset)` (it loads its own data).

In [11]:
preset = Preset(name='my_btc_support_break', scenario=SCENARIO, stop=STOP, tp=TP, secondary=SECONDARY,
                params=PARAMS, cfg=CFG,
                data=dict(symbol=SYMBOL, interval=TF, start=START, end=END, source=SOURCE,
                          market=MARKET, tz='UTC'),
                description='My tuned support-break short.')
path = preset.to_json('../scenarios/presets/my_btc_support_break.json')
print('saved ->', path)
r2 = sc.run(Preset.from_json(path))
print('reproduced:', int(r2.stats['num_trades']), 'trades | ret', round(r2.stats['total_return_pct'], 2), '%')

[17:51:38] INFO | quant.data | cache HIT  binance BTCUSDT 1m -> pushdown load
[17:51:38] INFO | quant.engine | backtest done | bars=2,881 trades=8 final_cash=10151.04 return=1.51% dd=0.54% | 0.8 ms


saved -> ..\scenarios\presets\my_btc_support_break.json
reproduced: 8 trades | ret 1.51 %


## 11 . Reading the setup
- **v1.0 core** — close below the EMA line + a support-**zone** break; one entry per episode (re-arms
  when price closes back above the EMA line).
- **v1.1 +structure** — also needs the prior HH/HL run into resistance (the topped-out shape).
- **v1.2 no-structure** — no S/R zones: N red candles below the EMA + a break of the prior-N-bar low.
- **v1.3 nearest-trough** — breaks the lower of two nearby troughs when the auto-zone is weak.

**On stops:** a pivot-based level (`swing_high`, `prev_high`) can never be fresher than its `lr`
confirmation bars — that is *why* a stop can sit further away than the high you see just before entry.
`leg_high` and `high_of_n` avoid that entirely. Section 7 makes the difference concrete.

> On a trending-up BTC window a short setup is *supposed* to get stopped often — that is the honest
> read, and the chart lets you see it. Try a topping/ranging window, or the `atr` + `trailing` combo.